# GLADA Ablation Study: Systematic Component Analysis
## Quantifying the Contribution of Each GLADA Component to Cross-Domain Generalization

**Author**: Jakob Kauffmann (MS CS, SJSU)  
**Advisor**: Dr. Faranak Abri  

---

### Purpose

This notebook orchestrates a **comprehensive ablation study** for the GLADA audio deepfake detection framework.
The goal is to isolate and quantify the contribution of each architectural component to cross-domain
generalization, measured primarily by Equal Error Rate (EER) on the In-the-Wild (ITW) dataset
(Muller et al., Interspeech 2022).

**All model architecture, training logic, dataset classes, and evaluation code lives in the `glada` Python
package**. This notebook is a **thin orchestration layer** that configures experiments, dispatches
training runs, and analyzes results.

### Ablation Design

We follow a multi-axis ablation strategy:

| Axis | Experiments | What We Learn |
|---|---|---|
| **Leave-one-out** | Remove exactly one component at a time | Per-component contribution ($\Delta$ from full GLADA) |
| **Key baselines** | IB-CAAN-like, HierCon-like, no DA, backbone-only | GLADA vs. prior work ablations |
| **Augmentation** | RawBoost, +MUSAN, +RIR, MUSAN+RIR only | Augmentation strategy impact |
| **Multi-backbone** | WavLM-Large, XLS-R, Data2Vec, Wav2Vec2-Robust, WavLM-Base-Plus | Backbone sensitivity |
| **Multi-seed** | Seeds 42, 123, 2026 on top 2 backbones | Statistical reliability |

### GLADA Loss Function

$$\mathcal{L} = \mathcal{L}_{\text{task}} + \alpha \cdot \mathcal{L}_{\text{GLADA}} + \beta \cdot \mathcal{L}_{\text{KL}} + \gamma \cdot \mathcal{L}_{\text{speaker}} + \delta \cdot \mathcal{L}_{\text{CKA}}$$

where:
- $\mathcal{L}_{\text{GLADA}} = \frac{1}{K} \sum_{k=1}^{K} \text{CE}(D_k(\text{GRL}(g_k, \alpha)), d)$ -- per-group domain adversarial loss
- $\mathcal{L}_{\text{KL}} = \text{KL}(q(z|x) \| \mathcal{N}(0, I))$ -- VIB compression (Alemi et al., ICLR 2017)
- $\mathcal{L}_{\text{speaker}} = \text{CE}(S(\text{GRL}(z, \alpha_{\text{spk}})), \text{spk})$ -- speaker disentanglement
- $\mathcal{L}_{\text{CKA}} = \frac{1}{\binom{K}{2}} \sum_{i<j} \text{CKA}(g_i, g_j)$ -- group diversity (Kornblith et al., ICML 2019)

Each ablation switch toggles one or more of these terms, isolating that component's effect on the total loss
and downstream generalization.

### Expected Runtime

With reduced epochs (3 + 17 = 20 total) on A100, each run takes approximately 2 hours.
The full ablation sweep (approx. 30 configs) requires approximately 60 GPU-hours. Crash-resume support
ensures that Colab disconnections do not require restarting the entire sweep.

## 1. Environment Setup

Install dependencies, mount Google Drive, extract datasets to Colab local SSD, and
optionally download MUSAN and RIR augmentation corpora.

| Corpus | Source | Size | Reference |
|---|---|---|---|
| ASVspoof 2019 LA | [ASVspoof Challenge](https://www.asvspoof.org/) | ~8 GB | Wang et al., Computer Speech & Language, 2020 |
| In-the-Wild (ITW) | [Muller et al.](https://doi.org/10.21437/Interspeech.2022-10078) | ~20 GB | Muller et al., Interspeech 2022 |
| MUSAN | [OpenSLR-17](https://openslr.org/17/) | ~11 GB | Snyder et al., 2015 |
| RIR | [OpenSLR-28](https://openslr.org/28/) | ~5 GB | Ko et al., Interspeech 2017 |

In [1]:
# Install dependencies (transformers for SSL backbones, soundfile for audio I/O)
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import tarfile

In [3]:
# --- Copy MUSAN and RIR corpora from Drive to local runtime ---
#
# MUSAN (Snyder et al., 2015): Music, Speech, and Noise corpus from OpenSLR-17.
#   Three noise categories: noise (ambient/environmental), speech (babble), music.
#   Each category has SNR ranges tuned for realistic mixing.
#d
# RIR (Ko et al., Interspeech 2017): Simulated room impulse responses from OpenSLR-28.
#   Models acoustic channel variability (room size, materials, reverberation time).
#
# These are used by CombinedAugmenter in the augmentation ablation experiments.
# Huang et al. (IB-CAAN, 2025) showed that combining RawBoost + MUSAN + RIR
# achieved 4.61% ITW EER vs. 4.93% with RawBoost alone.

import shutil, subprocess

MUSAN_PATH = '/content/data/musan'
RIR_PATH = '/content/data/RIRS_NOISES/simulated_rirs'
MUSAN_DRIVE_PATH = '/content/drive/MyDrive/Jakob/Datasets/musan.tar.gz'
RIR_DRIVE_PATH = '/content/drive/MyDrive/Jakob/Datasets/rirs_noises.zip'

os.makedirs('/content/data', exist_ok=True)

if not os.path.isdir(MUSAN_PATH):
    print(f'Copying MUSAN from Drive...')
    shutil.copy2(MUSAN_DRIVE_PATH, '/content/data/musan.tar.gz')
    subprocess.run(['tar', '-xzf', '/content/data/musan.tar.gz', '-C', '/content/data/'], check=True)
    os.remove('/content/data/musan.tar.gz')
    print(f'MUSAN extracted to {MUSAN_PATH}')
else:
    print(f'MUSAN already present at {MUSAN_PATH}')

if not os.path.isdir(RIR_PATH):
    print(f'Copying RIR from Drive...')
    shutil.copy2(RIR_DRIVE_PATH, '/content/data/rirs_noises.zip')
    subprocess.run(['unzip', '-q', '/content/data/rirs_noises.zip', '-d', '/content/data/'], check=True)
    os.remove('/content/data/rirs_noises.zip')
    print(f'RIR extracted to {RIR_PATH}')
else:
    print(f'RIR already present at {RIR_PATH}')

print(f'\nMUSAN ready: {os.path.isdir(MUSAN_PATH)}')
print(f'RIR ready:   {os.path.isdir(RIR_PATH)}')


Copying MUSAN from Drive...
MUSAN extracted to /content/data/musan
Copying RIR from Drive...
RIR extracted to /content/data/RIRS_NOISES/simulated_rirs

MUSAN ready: True
RIR ready:   True


In [4]:
def extract_dataset(tar_path, extract_to, name):
    """Extract a tar archive from Drive to local storage, skipping if already done."""
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}')
        return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t:
        t.extractall(extract_to)
    print(f'{name}: done.')

# --- Drive paths (tar archives) ---
DRIVE_ASVSPOOF_TAR = '/content/drive/MyDrive/Jakob/Datasets/LA.tar'
DRIVE_ITW_TAR = '/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar'

# --- Local extraction targets (Colab SSD for fast I/O) ---
extract_dataset(DRIVE_ASVSPOOF_TAR, '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset(DRIVE_ITW_TAR, '/content/data/itw', 'In-the-Wild')

# --- Verify core paths ---
ASV_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO = '/content/data/itw/release_in_the_wild'

for p in [ASV_PROTO, ASV_AUDIO, ITW_META]:
    assert os.path.exists(p), f'Missing: {p}'
print('All core paths verified.')

ASVspoof 2019 LA: extracting /content/drive/MyDrive/Jakob/Datasets/LA.tar -> /content/data/asvspoof ...


/tmp/ipykernel_716/1401112357.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(extract_to)


ASVspoof 2019 LA: done.
In-the-Wild: extracting /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar -> /content/data/itw ...
In-the-Wild: done.
All core paths verified.


## 2. Package Imports

The `glada` package is synced to Google Drive and loaded via `sys.path.insert`.
All model architecture, training loops, dataset classes, and evaluation logic
live in this package

Key imports for the ablation sweep:
- `GLADAConfig`: Central dataclass with ablation switches (`use_glada`, `use_vib`, etc.)
- `GLADATrainer`: Two-phase training loop with GRL scheduling
- `CombinedAugmenter`: Chained augmentation pipeline -- RawBoost (Tak et al., ICASSP 2022) $\to$ MUSAN (Snyder et al., 2015) $\to$ RIR (Ko et al., 2017)
- `MultiDatasetEvaluator`: Evaluation with bootstrap confidence intervals
- `average_checkpoints`: Top-$K$ checkpoint averaging (Polyak & Juditsky, 1992; Huang et al., 2025)

In [5]:
import sys
!cp -r /content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code .
sys.path.insert(0, '/content/glada_code')

# Core configuration and utilities
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed

# Dataset classes (audio loading uses soundfile.read(), never torchaudio.load())
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,  # Source domain (labeled + speaker IDs)
    ASVspoofEvalDataset,          # In-domain eval (ASVspoof 2019 LA eval set)
    InTheWildDataset,             # Target domain (unlabeled for DA) + eval (labeled)
    create_dataloaders,           # DataLoader factory with balanced sampling
)

# Augmentation: RawBoost (Tak et al., 2022), MUSAN (Snyder et al., 2015),
# RIR (Ko et al., 2017), and chained pipeline
from glada.augmentation import RawBoost, CombinedAugmenter

# Trainer: two-phase training loop with GRL scheduling
from glada.trainers import GLADATrainer

# Evaluation: EER with bootstrap CI, checkpoint averaging
from glada.evaluation import (
    MultiDatasetEvaluator,
    average_checkpoints,
    save_averaged_checkpoint,
)

# Model class (needed for checkpoint averaging)
from glada.models import GLADAModel

import copy
import gc
import json
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Publication-quality matplotlib defaults
matplotlib.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print(f'GLADA package loaded.')
print(f'Available backbones: {list(BACKBONE_REGISTRY.keys())}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
    print(f'BFloat16 support: {torch.cuda.is_bf16_supported()}')

GLADA package loaded.
Available backbones: ['wavlm-large', 'xlsr-300m', 'data2vec-large', 'wav2vec2-robust', 'wavlm-base-plus', 'xlsr-1b']
CUDA: True
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
BFloat16 support: True


## 3. Configuration Matrix

The ablation study is organized as a **configuration matrix**: each experiment is defined by a
name, a backbone, a random seed, and a dictionary of `GLADAConfig` overrides that deviate from
the full GLADA defaults.

### 3.1 Leave-One-Out Ablations

Each removes **exactly one** component from the full GLADA model, measuring the performance
drop $\Delta\text{EER} = \text{EER}_{\text{ablated}} - \text{EER}_{\text{full}}$.
A positive $\Delta$ means the removed component was helping.

| Config | Removed Component | What It Tests |
|---|---|---|
| `full_glada` | (none -- all ON) | Full GLADA reference |
| `no_glada` | Per-group DA $\to$ global DA | **Primary contribution**: is grouped-layer DA (MLADA, Fang et al. 2021; DAA, Ngo Trung et al. 2021) better than single-level DA? |
| `no_vib` | VIB bottleneck | Does information compression (Alemi et al., ICLR 2017) help domain-invariant features? |
| `no_speaker` | Speaker adversary | Does speaker disentanglement improve generalization? |
| `no_cka` | CKA diversity loss | Does enforcing diverse group representations (Tran et al., MM 2025) help? |
| `no_rawboost` | RawBoost augmentation | How much does waveform augmentation (Tak et al., ICASSP 2022) contribute? |
| `no_hier` | Hierarchical pooling | Does grouped-layer attention (Pan et al., Interspeech 2024) outperform flat aggregation? |

### 3.2 Key Baselines

These compare GLADA against configurations resembling prior work:

| Config | Resembles | Architecture |
|---|---|---|
| `global_da_only` | IB-CAAN (Huang et al., 2025) | Global DA + VIB, no per-group DA |
| `hierarchical_only` | HierCon (Liang et al., WWW 2026) | Grouped attention only, no DA/VIB/speaker |
| `no_da` | No adversarial training | No DA, no speaker adversary |
| `backbone_only` | Vanilla SSL fine-tuning | Just SSL backbone + linear classifier |

### 3.3 Augmentation Ablation

Huang et al. (IB-CAAN, 2025) showed that combining RawBoost + MUSAN + RIR achieved
4.61% ITW EER vs. 4.93% with RawBoost alone. The `CombinedAugmenter` chains stages
in physical signal order:

$$\text{source} \xrightarrow{\text{RawBoost}} \text{channel effects} \xrightarrow{\text{MUSAN}} \text{env. noise} \xrightarrow{\text{RIR}} \text{room acoustics}$$

| Config | RawBoost | MUSAN | RIR |
|---|---|---|---|
| `rawboost_only` | Yes | No | No |
| `rawboost_musan` | Yes | Yes | No |
| `rawboost_musan_rir` | Yes | Yes | Yes |
| `musan_rir_only` | No | Yes | Yes |

### 3.4 Multi-Backbone Sweep

Full GLADA tested across all five supported backbones. El-Kheir et al. (NAACL 2025)
showed that different SSL models encode deepfake information differently across layers,
so GLADA's per-group DA may benefit some backbones more than others.

### 3.5 Multi-Seed Runs

Seeds $\{42, 123, 2026\}$ on the top 2 backbones (WavLM-Large and XLS-R-300M)
to compute mean $\pm$ std for statistical reliability.

In [6]:
# ============================================================================
# CONFIGURATION MATRIX
#
# Each entry maps a config name to a dict of GLADAConfig field overrides.
# An empty dict {} means full GLADA with all defaults.
#
# The ablation switches in GLADAConfig (config.py) that control each component:
#   use_glada             -- Per-group domain discriminators (primary contribution)
#   use_global_domain_disc -- Single global domain discriminator (baseline)
#   use_vib               -- Variational Information Bottleneck
#   use_speaker_adversary -- Speaker disentanglement via GRL
#   use_cka_loss          -- CKA group diversity regularization
#   use_hierarchical      -- Grouped-layer attention pooling
#   use_rawboost          -- RawBoost waveform augmentation
# ============================================================================

# --- Epoch budget ---
# Reduced epochs for the ablation sweep (~2h per run on A100).
# Set USE_FULL_EPOCHS = True for final publication-quality results.
USE_FULL_EPOCHS = False
ABLATION_PHASE1 = 5 if USE_FULL_EPOCHS else 3
ABLATION_PHASE2 = 45 if USE_FULL_EPOCHS else 17

# --- Backbones ---
BACKBONES = [
    'wavlm-large',       # 24 layers, 1024 dim, 4 groups
    'xlsr-300m',         # 24 layers, 1024 dim, 4 groups
    'data2vec-large',    # 24 layers, 1024 dim, 4 groups
    'wav2vec2-robust',   # 24 layers, 1024 dim, 4 groups
    'wavlm-base-plus',  # 12 layers, 768 dim, 3 groups
]

# --- Seeds for multi-seed runs ---
SEEDS = [42, 123, 2026]

# --- Augmentation paths ---
# These will be empty strings if the corpora are not available,
# causing CombinedAugmenter to skip those stages gracefully.
MUSAN_AVAIL = MUSAN_PATH if os.path.isdir(MUSAN_PATH) else ''
RIR_AVAIL = RIR_PATH if os.path.isdir(RIR_PATH) else ''

# ==========================================================================
# 1. LEAVE-ONE-OUT ABLATIONS
#    Remove exactly one component at a time from the full GLADA model.
#    The "no_glada" ablation replaces per-group DA with a single global
#    domain discriminator to directly test the thesis contribution.
# ==========================================================================
ABLATION_CONFIGS = {
    # Full GLADA (reference -- all components ON)
    'full_glada': {},

    # Remove per-group DA, replace with global DA.
    # THIS IS THE MOST IMPORTANT ABLATION -- directly tests whether
    # grouped-layer DA outperforms single-level DA.
    # Precedent: MLADA (Fang et al., Neurocomputing 2021) showed per-layer
    # DA outperforms single-layer DA in vision tasks.
    'no_glada': {
        'use_glada': False,
        'use_global_domain_disc': True,
    },

    # Remove VIB bottleneck (uses deterministic projection instead).
    # Tests whether information compression (Alemi et al., ICLR 2017)
    # helps learn domain-invariant features.
    'no_vib': {
        'use_vib': False,
    },

    # Remove speaker adversary (no speaker disentanglement).
    # Tests whether removing speaker-specific information from the
    # representation improves cross-domain generalization.
    'no_speaker': {
        'use_speaker_adversary': False,
    },

    # Remove CKA diversity loss.
    # Tests whether enforcing diverse group representations
    # (Tran et al., ACM MM 2025) improves per-group DA.
    'no_cka': {
        'use_cka_loss': False,
    },

    # Remove RawBoost augmentation (Tak et al., ICASSP 2022).
    # Tests how much waveform-level augmentation contributes
    # when adversarial DA is already handling domain shift.
    'no_rawboost': {
        'use_rawboost': False,
        'rawboost_prob': 0.0,
    },

    # Remove hierarchical pooling (flat weighted sum instead).
    # Tests whether grouped-layer attention (Pan et al., Interspeech 2024)
    # outperforms simpler aggregation strategies.
    'no_hier': {
        'use_hierarchical': False,
    },
}

# ==========================================================================
# 2. KEY BASELINES
#    Configurations resembling prior work for direct comparison.
# ==========================================================================
BASELINE_CONFIGS = {
    # IB-CAAN-like: global DA + VIB, no per-group DA.
    # Reference: Huang et al. (2025), "IB-CAAN", arxiv 2509.23618.
    # Achieved 4.93% ITW EER with VIB + single-level DANN.
    'global_da_only': {
        'use_glada': False,
        'use_global_domain_disc': True,
        'use_vib': True,
    },

    # HierCon-like: grouped attention only, no DA/VIB/speaker.
    # Reference: Liang et al. (WWW 2026), "HierCon", arxiv 2602.01032.
    # Achieved 6.87% ITW EER with hierarchical contrastive learning.
    'hierarchical_only': {
        'use_glada': False,
        'use_global_domain_disc': False,
        'use_vib': False,
        'use_speaker_adversary': False,
    },

    # No adversarial training at all.
    # Tests the contribution of ALL adversarial components combined
    # (domain DA + speaker disentanglement).
    'no_da': {
        'use_glada': False,
        'use_global_domain_disc': False,
        'use_speaker_adversary': False,
    },

    # Backbone-only: just SSL + classifier (minimal baseline).
    # No DA, no VIB, no speaker adversary, no CKA, no RawBoost.
    # This is the floor: how good is the SSL backbone alone?
    'backbone_only': {
        'use_glada': False,
        'use_global_domain_disc': False,
        'use_vib': False,
        'use_speaker_adversary': False,
        'use_cka_loss': False,
        'use_rawboost': False,
        'rawboost_prob': 0.0,
    },
}

# ==========================================================================
# 3. AUGMENTATION ABLATION
#    Tests all combinations of RawBoost, MUSAN, and RIR augmentation.
#    The CombinedAugmenter pipeline chains stages in physical signal order:
#    RawBoost (channel effects) -> MUSAN (additive noise) -> RIR (room reverb).
#    Each stage fires independently with configurable probability.
# ==========================================================================
AUG_CONFIGS = {
    # RawBoost only (default -- no MUSAN/RIR)
    'rawboost_only': {
        'musan_path': '',
        'rir_path': '',
    },

    # RawBoost + MUSAN noise
    'rawboost_musan': {
        'musan_path': MUSAN_AVAIL,
        'rir_path': '',
    },

    # RawBoost + MUSAN + RIR (full augmentation pipeline)
    'rawboost_musan_rir': {
        'musan_path': MUSAN_AVAIL,
        'rir_path': RIR_AVAIL,
    },

    # MUSAN + RIR only (no RawBoost) -- tests RawBoost's unique contribution
    'musan_rir_only': {
        'use_rawboost': False,
        'rawboost_prob': 0.0,
        'musan_path': MUSAN_AVAIL,
        'rir_path': RIR_AVAIL,
    },
}


def make_config(name, backbone='wavlm-large', seed=42, overrides=None):
    """
    Create a GLADAConfig with the given overrides applied on top of defaults.

    This is the central config factory for the ablation sweep. It ensures:
      - Consistent epoch budget (reduced for screening, full for final runs)
      - Colab-safe settings (num_workers=0, always)
      - Proper experiment naming for checkpoint organization on Drive
      - __post_init__() is called after overrides to recompute derived fields
        (grad_accum_steps, total_epochs, num_groups, hidden_dim, etc.)

    Args:
        name: Human-readable experiment name (used in results JSON and run_dir).
        backbone: SSL backbone key from BACKBONE_REGISTRY.
        seed: Random seed for reproducibility.
        overrides: Dict of GLADAConfig field names to override values.
            Empty dict {} means full GLADA with all components ON.

    Returns:
        A fully-configured GLADAConfig instance ready for GLADATrainer.
    """
    overrides = overrides or {}

    config = GLADAConfig(
        experiment_name=f'ablation_{name}',
        backbone=backbone,
        seed=seed,
        phase1_epochs=ABLATION_PHASE1,
        phase2_epochs=ABLATION_PHASE2,
        batch_size=24,
        num_workers=0,           # Always 0 on Colab
        save_every_n_epochs=1,   # Save every epoch for checkpoint averaging
        save_top_k=10,           # Keep top 10 periodic checkpoints
    )

    # Apply overrides
    for key, value in overrides.items():
        if hasattr(config, key):
            setattr(config, key, value)
        else:
            print(f'  WARNING: Unknown config key "{key}" in overrides for {name}')

    # Recompute derived fields (grad_accum_steps, total_epochs, etc.)
    config.__post_init__()

    return config


# --- Print configuration summary ---
total_experiments = (
    len(ABLATION_CONFIGS)
    + len(BASELINE_CONFIGS)
    + len(BACKBONES) - 1            # multi-backbone (-1 for wavlm-large already in ablation)
    + 2 * len(SEEDS) - 1            # multi-seed (-1 for wavlm-large seed 42 already covered)
    + len(AUG_CONFIGS)
)
epochs_per_run = ABLATION_PHASE1 + ABLATION_PHASE2
print(f'Ablation sweep configuration:')
print(f'  Leave-one-out:     {len(ABLATION_CONFIGS)} configs')
print(f'  Baselines:         {len(BASELINE_CONFIGS)} configs')
print(f'  Multi-backbone:    {len(BACKBONES) - 1} new configs (+1 already in leave-one-out)')
print(f'  Multi-seed:        {2 * len(SEEDS) - 1} new configs')
print(f'  Augmentation:      {len(AUG_CONFIGS)} configs')
print(f'  ---')
print(f'  Total experiments: {total_experiments}')
print(f'  Epochs per run:    {epochs_per_run} (Phase 1: {ABLATION_PHASE1}, Phase 2: {ABLATION_PHASE2})')
print(f'  Full epochs mode:  {"YES" if USE_FULL_EPOCHS else "NO (set USE_FULL_EPOCHS=True for final runs)"}')

Ablation sweep configuration:
  Leave-one-out:     7 configs
  Baselines:         4 configs
  Multi-backbone:    4 new configs (+1 already in leave-one-out)
  Multi-seed:        5 new configs
  Augmentation:      4 configs
  ---
  Total experiments: 24
  Epochs per run:    20 (Phase 1: 3, Phase 2: 17)
  Full epochs mode:  NO (set USE_FULL_EPOCHS=True for final runs)


## 4. Dataset Setup

Datasets are loaded **once** and shared across all configurations in the sweep.
Only the DataLoaders and augmenters are rebuilt per-config when augmentation settings differ.

### Domain Adaptation Setup (Ganin et al., JMLR 2016)

GLADA uses **unsupervised domain adaptation**:
- **Source domain** (ASVspoof 2019 LA train): Labeled data with task labels + speaker IDs
- **Target domain** (ITW): Unlabeled -- only domain membership ($d \in \{0, 1\}$) is used by the GRL

The domain discriminators receive gradient-reversed features and learn to distinguish
source ($d=0$) from target ($d=1$). By reversing these gradients, the feature extractor
is trained to produce domain-invariant representations -- the core mechanism for
cross-domain generalization.

### Balanced Sampling

ASVspoof 2019 LA has a severe class imbalance (~90% spoof). We use `WeightedRandomSampler`
with inverse class frequency weights to ensure approximately 50/50 class balance in each batch.

In [7]:
# --- Create a base config for dataset construction ---
# Datasets need a config for max_audio_len and num_workers.
# Augmentation is handled separately per-config in the sweep loop via
# CombinedAugmenter, which reads augmentation settings from the config.
base_config = GLADAConfig(
    batch_size=24,
    num_workers=0,  # Always 0 on Colab
)

print('Loading datasets (shared across all ablation configs)...')

# Source domain: ASVspoof 2019 LA training set (labeled + speaker IDs).
# Audio is loaded via soundfile.read() at 16 kHz, padded/truncated to 4s.
# NOTE: augmenter is set to None here; it will be configured per-run.
source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=base_config,
    augmenter=None,  # Set per-config in sweep loop
)

# Target domain: ITW (unlabeled for adversarial adaptation).
# use_labels=False: only domain membership is used by GRL, not task labels.
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=base_config,
    use_labels=False,
)

# Evaluation: ITW with ground truth labels for EER computation.
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=base_config,
    use_labels=True,
)

# Optional: ASVspoof 2019 LA eval set (in-domain evaluation).
extra_eval = {}
if os.path.exists(ASV_EVAL_PROTO) and os.path.exists(ASV_EVAL_AUDIO):
    extra_eval['2019_LA'] = ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO,
        audio_dir=ASV_EVAL_AUDIO,
        config=base_config,
    )

# Detect number of speakers from source dataset (used for speaker adversary head)
num_speakers = None
if hasattr(source_ds, 'speaker_to_id'):
    num_speakers = len(source_ds.speaker_to_id)

print(f'\nSource: {len(source_ds):,} samples, {num_speakers} speakers')
print(f'Target: {len(target_ds):,} samples (unlabeled)')
print(f'Eval:   {len(eval_ds):,} samples (ITW, labeled)')
for name, ds in extra_eval.items():
    print(f'Extra:  {len(ds):,} samples ({name})')

Loading datasets (shared across all ablation configs)...
  ASVspoof: 25380 samples, 20 speakers, rawboost=0.5
  ITW: 31779 samples (unlabeled returned)
  ITW: 31779 samples (labeled returned)
  ASVspoof Eval: 71237 samples

Source: 25,380 samples, 20 speakers
Target: 31,779 samples (unlabeled)
Eval:   31,779 samples (ITW, labeled)
Extra:  71,237 samples (2019_LA)


## 5. Ablation Sweep

### Sweep Design

The sweep runs all experiments sequentially, with **crash-resume support**:
- Results are saved to `ablation_results.json` on Google Drive after **each** run
- On restart, completed experiments are detected and skipped automatically
- Per-run exceptions are caught and logged without aborting the entire sweep
- GPU memory is cleaned between runs to prevent OOM accumulation

### Experiment Execution Order

1. **Leave-one-out + baselines** on WavLM-Large (seed 42) -- the primary ablation table
2. **Full GLADA across all backbones** (seed 42) -- backbone comparison
3. **Multi-seed** for WavLM-Large and XLS-R-300M (seeds 42, 123, 2026) -- statistical reliability
4. **Augmentation ablation** on WavLM-Large (seed 42) -- augmentation strategy

### Augmenter Handling

When augmentation settings change between configs, the source dataset's augmenter is
updated via a fresh `CombinedAugmenter` instance. The pipeline chains up to three stages:

$$\text{RawBoost} \xrightarrow{p=0.5} \text{MUSAN} \xrightarrow{p=0.5} \text{RIR} \xrightarrow{p=0.5}$$

Each stage fires independently with its configured probability, producing $2^3 = 8$
possible augmentation states per sample.

In [8]:
# ============================================================================
# BUILD THE FULL EXPERIMENT LIST
#
# Phase 1 (complete): Leave-one-out + baselines on wavlm-base-plus.
# Phase 2 (this run): Multi-backbone, multi-seed, augmentation ablation.
# Crash-resume will auto-skip all 11 completed Phase 1 experiments.
# ============================================================================

experiments = []

# --- 1. Leave-one-out ablations on BASE-PLUS (Phase 1 -- will be skipped) ---
for name, overrides in ABLATION_CONFIGS.items():
    experiments.append((name, 'wavlm-base-plus', 42, overrides))

# --- 2. Key baselines on BASE-PLUS (Phase 1 -- will be skipped) ---
for name, overrides in BASELINE_CONFIGS.items():
    experiments.append((name, 'wavlm-base-plus', 42, overrides))

# --- 3. Multi-backbone: full GLADA on each backbone ---
for bb in BACKBONES:
    if bb == 'wavlm-base-plus':
        continue
    experiments.append((f'full_{bb}', bb, 42, {}))

# --- 4. Multi-seed: statistical reliability on top-2 backbones ---
for bb in ['wavlm-large', 'xlsr-300m']:
    for seed in SEEDS:
        if bb == 'wavlm-large' and seed == 42:
            continue
        experiments.append((f'full_{bb}_s{seed}', bb, seed, {}))

# --- 5. Augmentation ablation on wavlm-large ---
for name, overrides in AUG_CONFIGS.items():
    experiments.append((f'aug_{name}', 'wavlm-large', 42, overrides))

print(f'Total experiments in sweep: {len(experiments)}')
print(f'\nExperiment list:')
for i, (name, bb, seed, ovr) in enumerate(experiments):
    ovr_str = ', '.join(f'{k}={v}' for k, v in ovr.items()) if ovr else '(full GLADA)'
    print(f'  {i+1:3d}. {name:<30s}  bb={bb:<20s}  seed={seed}  {ovr_str}')


Total experiments in sweep: 24

Experiment list:
    1. full_glada                      bb=wavlm-base-plus       seed=42  (full GLADA)
    2. no_glada                        bb=wavlm-base-plus       seed=42  use_glada=False, use_global_domain_disc=True
    3. no_vib                          bb=wavlm-base-plus       seed=42  use_vib=False
    4. no_speaker                      bb=wavlm-base-plus       seed=42  use_speaker_adversary=False
    5. no_cka                          bb=wavlm-base-plus       seed=42  use_cka_loss=False
    6. no_rawboost                     bb=wavlm-base-plus       seed=42  use_rawboost=False, rawboost_prob=0.0
    7. no_hier                         bb=wavlm-base-plus       seed=42  use_hierarchical=False
    8. global_da_only                  bb=wavlm-base-plus       seed=42  use_glada=False, use_global_domain_disc=True, use_vib=True
    9. hierarchical_only               bb=wavlm-base-plus       seed=42  use_glada=False, use_global_domain_disc=False, use_vib=

In [ ]:
# ============================================================================
# ABLATION SWEEP LOOP
#
# Key features:
#   - Crash-resume: loads existing results JSON, skips completed configs
#   - Per-run exception handling (catches errors without aborting the sweep)
#   - Results saved after EACH run to Google Drive for persistence
#   - GPU memory cleaned between runs (del trainer + empty_cache + gc.collect)
#   - CombinedAugmenter updated per-config based on augmentation overrides
# ============================================================================

ABLATION_RESULTS_DIR = '/content/drive/MyDrive/Jakob/Experiments/ablation'
RESULTS_FILE = os.path.join(ABLATION_RESULTS_DIR, 'ablation_results.json')
os.makedirs(ABLATION_RESULTS_DIR, exist_ok=True)

# --- Load existing results for crash-resume ---
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} existing results from {RESULTS_FILE}')
else:
    all_results = {}
    print(f'Starting fresh ablation sweep.')

sweep_start = time.time()
total = len(experiments)
completed = 0
skipped = 0

for i, (exp_name, backbone, seed, overrides) in enumerate(experiments):

    # --- Crash-resume: skip completed experiments ---
    if exp_name in all_results and 'error' not in all_results[exp_name]:
        skipped += 1
        print(f'\n[{i+1}/{total}] SKIP (already complete): {exp_name}')
        continue

    elapsed = time.time() - sweep_start
    eta = (elapsed / max(completed, 1)) * (total - completed - skipped) if completed > 0 else 0

    print(f'\n{"="*60}')
    print(f'[{i+1}/{total}] {exp_name}')
    print(f'  backbone={backbone}, seed={seed}')
    print(f'  overrides: {overrides if overrides else "(full GLADA)"}')
    print(f'  Elapsed: {elapsed/60:.1f} min | Completed: {completed} | Skipped: {skipped} | ETA: {eta/60:.1f} min')
    print(f'{"="*60}')

    try:
        # --- Build config ---
        config = make_config(exp_name, backbone=backbone, seed=seed, overrides=overrides)
        set_seed(seed)

        # --- Update augmenter based on config ---
        augmenter = CombinedAugmenter(config)
        source_ds.augmenter = augmenter
        print(f'  Augmenter: {augmenter}')

        # --- Create DataLoaders ---
        loaders = create_dataloaders(
            config=config,
            source_ds=source_ds,
            target_ds=target_ds,
            eval_ds=eval_ds,
            extra_eval=extra_eval,
        )

        # --- Build trainer and train ---
        trainer = GLADATrainer(
            config=config,
            source_loader=loaders['source'],
            target_loader=loaders['target'],
            eval_loader=loaders['eval'],
            extra_eval_loaders={k: v for k, v in loaders.items()
                                if k not in ('source', 'target', 'eval')},
            num_speakers=num_speakers,
        )

        results = trainer.train()

        # Store results (exclude full history to keep JSON manageable)
        all_results[exp_name] = {
            'best_eer': results['best_eer'],
            'best_epoch': results['best_epoch'],
            'run_dir': results['run_dir'],
            'backbone': backbone,
            'seed': seed,
            'overrides': overrides,
            'total_epochs': config.total_epochs,
        }

        completed += 1
        print(f'  DONE: best EER = {results["best_eer"]:.2f}% (epoch {results["best_epoch"]})')

    except Exception as e:
        print(f'  FAILED: {e}')
        import traceback
        traceback.print_exc()
        all_results[exp_name] = {
            'error': str(e),
            'backbone': backbone,
            'seed': seed,
            'overrides': overrides,
        }

    # --- Save results after EACH run (crash-resume support) ---
    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_results, f, indent=2, default=str)

    # --- GPU cleanup between runs ---
    try:
        del trainer
    except NameError:
        pass
    try:
        del results
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

# --- Final summary ---
total_time = (time.time() - sweep_start) / 60
n_failed = total - completed - skipped
print(f'\n{"="*60}')
print(f'ABLATION SWEEP COMPLETE')
print(f'  Completed: {completed} | Skipped (already done): {skipped} | Failed: {n_failed}')
print(f'  Total time: {total_time:.1f} min')
print(f'  Results: {RESULTS_FILE}')
print(f'{"="*60}')

Loaded 14 existing results from /content/drive/MyDrive/Jakob/Experiments/ablation/ablation_results.json

[1/24] SKIP (already complete): full_glada

[2/24] SKIP (already complete): no_glada

[3/24] SKIP (already complete): no_vib

[4/24] SKIP (already complete): no_speaker

[5/24] SKIP (already complete): no_cka

[6/24] SKIP (already complete): no_rawboost

[7/24] SKIP (already complete): no_hier

[8/24] SKIP (already complete): global_da_only

[9/24] SKIP (already complete): hierarchical_only

[10/24] SKIP (already complete): no_da

[11/24] SKIP (already complete): backbone_only

[12/24] SKIP (already complete): full_wavlm-large

[13/24] SKIP (already complete): full_xlsr-300m

[14/24] SKIP (already complete): full_data2vec-large

[15/24] full_wav2vec2-robust
  backbone=wav2vec2-robust, seed=42
  overrides: (full GLADA)
  Elapsed: 0.0 min | Completed: 0 | Skipped: 14 | ETA: 0.0 min
  Augmenter: CombinedAugmenter(RawBoost(p=0.5))
  Updating num_speakers: 107 -> 20

Building GLADA model

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-robust
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable: 12,975,163 / 328,413,883
  Run dir: /content/drive/MyDrive/Jakob/Experiments/ablation_full_wav2vec2-robust/20260306_022041

GLADA Training | wav2vec2-robust | seed=42
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.001, SPK=0.1, CKA=0.01


... (249 outputs left out)

Ep 6:   0%|          | 0/1057 [00:00<?, ?it/s]

  [WARN] KL clamped 5000x consecutive (raw=9.524e+05, clamp=10.0)
  [WARN] KL clamped 6000x consecutive (raw=1.209e+06, clamp=10.0)
  [ADAPT] VIB beta halved: 0.000125 -> 6.25e-05 (KL raw avg=827346.5 > 1000)
  Ep 6 P2: L=0.208 KL=10.00 | ITW=19.30% | 2019_LA=0.64% BEST


Ep 7:   0%|          | 0/1057 [00:00<?, ?it/s]

  [WARN] KL clamped 7000x consecutive (raw=4.386e+05, clamp=10.0)
  [ADAPT] VIB beta halved: 6.25e-05 -> 3.125e-05 (KL raw avg=838214.8 > 1000)


## 6. Checkpoint Averaging

After all runs complete, we apply **top-5 checkpoint averaging** to the best configs.
This is a standard post-training technique that yields "free" generalization gains
without additional training cost.

### Theoretical Justification

- **Polyak & Juditsky (1992)**: Averaging iterates of SGD converges at the optimal
  $O(1/N)$ rate. For parameters in the same loss basin, the average lies closer to
  the basin center, which tends to be flatter.

- **Izmailov et al. (2018), SWA**: Weight averaging finds solutions in flatter regions
  of the loss landscape, improving generalization.

- **Huang et al. (IB-CAAN, 2025)**: Applied top-5 checkpoint averaging to audio
  deepfake detection, improving ITW EER from 4.93% to 4.61%.

### Averaging Formula

$$\theta_{\text{avg}} = \frac{1}{K} \sum_{i=1}^{K} \theta_i$$

where $\theta_i$ is the parameter vector of the $i$-th best checkpoint (ranked by
validation EER), and $K=5$ following Huang et al. (2025).

In [ ]:
# ============================================================================
# CHECKPOINT AVERAGING ON BEST CONFIGS
# ============================================================================

with open(RESULTS_FILE, 'r') as f:
    all_results = json.load(f)

CONFIGS_TO_AVERAGE = ['full_glada', 'no_glada']

# Also average the best non-wavlm-large backbone result
backbone_results = {
    name: r for name, r in all_results.items()
    if name.startswith('full_') and 'error' not in r
    and name not in ('full_glada',)
    and not name.endswith(('_s42', '_s123', '_s2026'))
}
if backbone_results:
    best_bb_name = min(backbone_results, key=lambda n: backbone_results[n].get('best_eer', 999))
    CONFIGS_TO_AVERAGE.append(best_bb_name)
    print(f'Also averaging best backbone: {best_bb_name}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
averaged_results = {}

for config_name in CONFIGS_TO_AVERAGE:
    if config_name not in all_results or 'error' in all_results[config_name]:
        print(f'\nSkipping {config_name}: not found or failed.')
        continue

    result = all_results[config_name]
    run_dir = result.get('run_dir', '')

    if not run_dir or not os.path.exists(run_dir):
        print(f'\nSkipping {config_name}: run_dir not found ({run_dir})')
        continue

    print(f'\n{"="*40}')
    print(f'Averaging checkpoints for: {config_name}')
    print(f'  Run dir: {run_dir}')
    print(f'  Best single-ckpt EER: {result["best_eer"]:.2f}%')

    try:
        config_path = os.path.join(run_dir, 'config.json')
        with open(config_path, 'r') as f:
            saved_cfg = json.load(f)
        cfg = GLADAConfig(**{k: v for k, v in saved_cfg.items()
                             if k in GLADAConfig.__dataclass_fields__})

        avg_model = average_checkpoints(
            run_dir=run_dir,
            model_class=GLADAModel,
            config=cfg,
            top_k=5,
            metric='eer',
            device=str(device),
        )

        all_eval_loaders = create_dataloaders(cfg, source_ds, target_ds, eval_ds, extra_eval)
        eval_loaders_dict = {'ITW': all_eval_loaders['eval']}
        for name in extra_eval:
            if name in all_eval_loaders:
                eval_loaders_dict[name] = all_eval_loaders[name]

        evaluator = MultiDatasetEvaluator(avg_model, cfg, device, eval_loaders_dict)
        eval_results = evaluator.evaluate_all()

        save_averaged_checkpoint(
            model=avg_model, run_dir=run_dir, config=cfg,
            top_k=5, source_epochs=[],
        )

        averaged_results[config_name] = {
            'single_best_eer': result['best_eer'],
            'averaged_eer': eval_results.get('ITW', {}).get('eer', None),
            'eval_results': {
                k: {kk: float(vv) if hasattr(vv, 'item') else vv
                    for kk, vv in v.items()}
                for k, v in eval_results.items()
            },
        }

        avg_eer = eval_results.get('ITW', {}).get('eer', float('nan'))
        delta = avg_eer - result['best_eer']
        print(f'  Averaged EER: {avg_eer:.2f}% (delta: {delta:+.2f}%)')

    except Exception as e:
        print(f'  FAILED: {e}')
        import traceback
        traceback.print_exc()

    try:
        del avg_model
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

averaged_results_path = os.path.join(ABLATION_RESULTS_DIR, 'averaged_results.json')
with open(averaged_results_path, 'w') as f:
    json.dump(averaged_results, f, indent=2, default=str)
print(f'\nAveraged results saved to {averaged_results_path}')

## 7. Results Analysis

We now load all results and generate the ablation tables and figures required for the
thesis defense.

### Output Tables

1. **Leave-one-out ablation table**: Per-component $\Delta$EER from full GLADA
2. **Backbone comparison table**: Full GLADA across all 5 backbones
3. **Multi-seed table**: Mean $\pm$ std for the top 2 backbones
4. **Augmentation comparison table**: RawBoost vs. MUSAN vs. RIR combinations

### Statistical Reporting

Two complementary measures of uncertainty:

- **Multi-seed variance** (seeds 42, 123, 2026): Measures training stochasticity
- **Bootstrap 95% CI** (1000 resamples): Measures evaluation variance

In [ ]:
# ============================================================================
# LOAD RESULTS AND BUILD DATAFRAME
# ============================================================================

with open(RESULTS_FILE, 'r') as f:
    all_results = json.load(f)

rows = []
for name, r in all_results.items():
    if 'error' in r:
        rows.append({'config': name, 'backbone': r.get('backbone', ''),
                     'seed': r.get('seed', ''), 'best_eer': None,
                     'best_epoch': None, 'status': f'FAILED: {r["error"][:60]}'})
    else:
        rows.append({'config': name, 'backbone': r.get('backbone', ''),
                     'seed': r.get('seed', ''), 'best_eer': r.get('best_eer'),
                     'best_epoch': r.get('best_epoch'), 'status': 'OK'})

df = pd.DataFrame(rows)
print(f'Total experiments: {len(df)}')
print(f'Successful: {(df["status"] == "OK").sum()}')
print(f'Failed: {(df["status"] != "OK").sum()}')
print()

df_ok = df[df['status'] == 'OK'].sort_values('best_eer')
print('All successful results (sorted by EER):')
print(df_ok[['config', 'backbone', 'seed', 'best_eer', 'best_epoch']].to_string(index=False))

In [ ]:
# ============================================================================
# TABLE 1: LEAVE-ONE-OUT ABLATION
# ============================================================================

full_eer = all_results.get('full_glada', {}).get('best_eer')

if full_eer is not None:
    ablation_rows = [
        ('full_glada',  'Full GLADA',       'All components ON (reference)'),
        ('no_glada',    'w/o Per-Group DA',  'Global DA instead of per-group DA'),
        ('no_vib',      'w/o VIB',           'Deterministic projection instead of VIB'),
        ('no_speaker',  'w/o Speaker Adv.',  'No speaker disentanglement'),
        ('no_cka',      'w/o CKA Loss',      'No group diversity regularization'),
        ('no_rawboost', 'w/o RawBoost',      'No waveform augmentation'),
        ('no_hier',     'w/o Hierarchical',  'Flat pooling instead of grouped attention'),
    ]

    print('=' * 85)
    print('TABLE 1: Leave-One-Out Ablation (WavLM-Large, seed 42)')
    print('=' * 85)
    print(f'{"Config":<25s}  {"ITW EER (%)":>12s}  {"Delta":>8s}  Description')
    print('-' * 85)

    for key, label, desc in ablation_rows:
        eer = all_results.get(key, {}).get('best_eer')
        if eer is not None:
            delta = eer - full_eer
            delta_str = f'{delta:+.2f}' if key != 'full_glada' else '--'
            print(f'{label:<25s}  {eer:>12.2f}  {delta_str:>8s}  {desc}')
        else:
            print(f'{label:<25s}  {"N/A":>12s}  {"N/A":>8s}  {desc}')

    print()
    print('Key Baselines:')
    print('-' * 85)
    baseline_rows = [
        ('global_da_only',    'IB-CAAN-like',    'Global DA + VIB (Huang et al., 2025)'),
        ('hierarchical_only', 'HierCon-like',    'Grouped attention only (Liang et al., 2026)'),
        ('no_da',             'No DA',           'No adversarial training'),
        ('backbone_only',     'Backbone Only',   'SSL + classifier only (floor)'),
    ]
    for key, label, desc in baseline_rows:
        eer = all_results.get(key, {}).get('best_eer')
        if eer is not None:
            delta = eer - full_eer
            print(f'{label:<25s}  {eer:>12.2f}  {delta:>+8.2f}  {desc}')
        else:
            print(f'{label:<25s}  {"N/A":>12s}  {"N/A":>8s}  {desc}')
    print('=' * 85)
else:
    print('WARNING: full_glada result not found. Cannot compute deltas.')

In [ ]:
# ============================================================================
# TABLE 2: BACKBONE COMPARISON
# ============================================================================

backbone_configs = [
    ('full_glada',           'WavLM-Large',      'wavlm-large'),
    ('full_xlsr-300m',       'XLS-R-300M',       'xlsr-300m'),
    ('full_data2vec-large',  'Data2Vec-Large',    'data2vec-large'),
    ('full_wav2vec2-robust', 'Wav2Vec2-Robust',   'wav2vec2-robust'),
    ('full_wavlm-base-plus', 'WavLM-Base-Plus',  'wavlm-base-plus'),
]

print('=' * 75)
print('TABLE 2: Backbone Comparison (Full GLADA, seed 42)')
print('=' * 75)
print(f'{"Backbone":<20s}  {"Layers":>6s}  {"Groups":>6s}  {"Dim":>5s}  {"ITW EER (%)":>12s}  {"Epoch":>6s}')
print('-' * 75)

for key, label, bb_key in backbone_configs:
    bb_info = BACKBONE_REGISTRY[bb_key]
    eer = all_results.get(key, {}).get('best_eer')
    epoch = all_results.get(key, {}).get('best_epoch')
    if eer is not None:
        print(f'{label:<20s}  {bb_info["num_layers"]:>6d}  {bb_info["num_groups"]:>6d}  '
              f'{bb_info["dim"]:>5d}  {eer:>12.2f}  {epoch:>6d}')
    else:
        print(f'{label:<20s}  {bb_info["num_layers"]:>6d}  {bb_info["num_groups"]:>6d}  '
              f'{bb_info["dim"]:>5d}  {"N/A":>12s}  {"N/A":>6s}')
print('=' * 75)

In [ ]:
# ============================================================================
# TABLE 3: MULTI-SEED RESULTS
# ============================================================================

print('=' * 65)
print('TABLE 3: Multi-Seed Results (Mean +/- Std)')
print('=' * 65)
print(f'{"Backbone":<20s}  {"Seeds":>15s}  {"Mean EER (%)":>12s}  {"Std":>6s}')
print('-' * 65)

for bb in ['wavlm-large', 'xlsr-300m']:
    seed_eers, found_seeds = [], []
    for seed in SEEDS:
        key = f'full_{bb}_s{seed}'
        if bb == 'wavlm-large' and seed == 42:
            key = 'full_glada'
        eer = all_results.get(key, {}).get('best_eer')
        if eer is not None:
            seed_eers.append(eer)
            found_seeds.append(seed)

    bb_label = 'WavLM-Large' if bb == 'wavlm-large' else 'XLS-R-300M'
    if len(seed_eers) >= 2:
        mean_eer = np.mean(seed_eers)
        std_eer = np.std(seed_eers)
        seeds_str = ', '.join(str(s) for s in found_seeds)
        print(f'{bb_label:<20s}  {seeds_str:>15s}  {mean_eer:>12.2f}  {std_eer:>6.2f}')
        for s, e in zip(found_seeds, seed_eers):
            print(f'  seed {s}:                                   {e:>12.2f}')
    elif len(seed_eers) == 1:
        print(f'{bb_label:<20s}  {found_seeds[0]:>15d}  {seed_eers[0]:>12.2f}  {"N/A":>6s}  (only 1 seed)')
    else:
        print(f'{bb_label:<20s}  {"N/A":>15s}  {"N/A":>12s}  {"N/A":>6s}')
print('=' * 65)

In [ ]:
# ============================================================================
# TABLE 4: AUGMENTATION COMPARISON
# ============================================================================

aug_display = [
    ('aug_rawboost_only',        'RawBoost Only',       'Y', '-', '-'),
    ('aug_rawboost_musan',       'RawBoost + MUSAN',    'Y', 'Y', '-'),
    ('aug_rawboost_musan_rir',   'RawBoost+MUSAN+RIR',  'Y', 'Y', 'Y'),
    ('aug_musan_rir_only',       'MUSAN + RIR Only',    '-', 'Y', 'Y'),
]

print('=' * 75)
print('TABLE 4: Augmentation Comparison (WavLM-Large, seed 42)')
print('=' * 75)
print(f'{"Config":<25s}  {"RB":>3s}  {"MU":>3s}  {"RIR":>3s}  {"ITW EER (%)":>12s}')
print('-' * 75)

for key, label, rb, mu, rir in aug_display:
    eer = all_results.get(key, {}).get('best_eer')
    if eer is not None:
        print(f'{label:<25s}  {rb:>3s}  {mu:>3s}  {rir:>3s}  {eer:>12.2f}')
    else:
        print(f'{label:<25s}  {rb:>3s}  {mu:>3s}  {rir:>3s}  {"N/A":>12s}')
print('=' * 75)

In [ ]:
# ============================================================================
# FIGURE 1: ABLATION BAR CHART (primary thesis figure)
# ============================================================================

bar_configs = [
    ('full_glada',        'Full\nGLADA',      'full'),
    ('no_glada',          'w/o\nPer-Group DA', 'ablation'),
    ('no_vib',            'w/o\nVIB',          'ablation'),
    ('no_speaker',        'w/o\nSpeaker',      'ablation'),
    ('no_cka',            'w/o\nCKA',          'ablation'),
    ('no_rawboost',       'w/o\nRawBoost',     'ablation'),
    ('no_hier',           'w/o\nHier.',        'ablation'),
    ('global_da_only',    'IB-CAAN\n-like',    'baseline'),
    ('hierarchical_only', 'HierCon\n-like',    'baseline'),
    ('no_da',             'No DA',             'baseline'),
    ('backbone_only',     'Backbone\nOnly',    'baseline'),
]

labels, eers, colors = [], [], []
color_map = {'full': '#2ecc71', 'ablation': '#3498db', 'baseline': '#e74c3c'}

for key, label, cat in bar_configs:
    eer = all_results.get(key, {}).get('best_eer')
    if eer is not None:
        labels.append(label)
        eers.append(eer)
        colors.append(color_map[cat])

if eers:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(labels))
    bars = ax.bar(x, eers, color=colors, edgecolor='white', linewidth=0.5, width=0.7)

    for bar_obj, eer_val in zip(bars, eers):
        ax.text(bar_obj.get_x() + bar_obj.get_width() / 2, bar_obj.get_height() + 0.2,
                f'{eer_val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    if full_eer is not None:
        ax.axhline(y=full_eer, color='#2ecc71', linestyle='--', alpha=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('ITW EER (%)')
    ax.set_title('GLADA Ablation Study: Per-Component Contribution to Cross-Domain Generalization')
    ax.set_ylim(0, max(eers) * 1.2)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=color_map['full'], label='Full GLADA'),
        Patch(facecolor=color_map['ablation'], label='Leave-one-out'),
        Patch(facecolor=color_map['baseline'], label='Baselines'),
    ]
    ax.legend(handles=legend_elements, loc='upper left')

    plt.tight_layout()
    fig_path = os.path.join(ABLATION_RESULTS_DIR, 'ablation_bar_chart.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f'Figure saved to {fig_path}')
    plt.show()
else:
    print('No results available for bar chart.')

In [ ]:
# ============================================================================
# FIGURE 2: BACKBONE COMPARISON BAR CHART
# ============================================================================

bb_labels, bb_eers = [], []
for key, label, bb_key in backbone_configs:
    eer = all_results.get(key, {}).get('best_eer')
    if eer is not None:
        bb_labels.append(label)
        bb_eers.append(eer)

if bb_eers:
    fig, ax = plt.subplots(figsize=(10, 5))
    best_idx = np.argmin(bb_eers)
    bb_colors = ['#3498db'] * len(bb_eers)
    bb_colors[best_idx] = '#2ecc71'
    bars = ax.bar(range(len(bb_labels)), bb_eers, color=bb_colors,
                  edgecolor='white', linewidth=0.5, width=0.6)
    for bar_obj, eer_val in zip(bars, bb_eers):
        ax.text(bar_obj.get_x() + bar_obj.get_width() / 2, bar_obj.get_height() + 0.15,
                f'{eer_val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(bb_labels)))
    ax.set_xticklabels(bb_labels, rotation=15, ha='right')
    ax.set_ylabel('ITW EER (%)')
    ax.set_title('Full GLADA: Backbone Comparison (seed 42)')
    ax.set_ylim(0, max(bb_eers) * 1.25)
    plt.tight_layout()
    fig_path = os.path.join(ABLATION_RESULTS_DIR, 'backbone_comparison.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f'Figure saved to {fig_path}')
    plt.show()
else:
    print('No backbone results available.')

In [ ]:
# ============================================================================
# FIGURE 3: MULTI-SEED BOX PLOT
# ============================================================================

seed_data = {}
for bb in ['wavlm-large', 'xlsr-300m']:
    eers_list = []
    for seed in SEEDS:
        key = f'full_{bb}_s{seed}'
        if bb == 'wavlm-large' and seed == 42:
            key = 'full_glada'
        eer = all_results.get(key, {}).get('best_eer')
        if eer is not None:
            eers_list.append(eer)
    if eers_list:
        bb_label = 'WavLM-Large' if bb == 'wavlm-large' else 'XLS-R-300M'
        seed_data[bb_label] = eers_list

if seed_data:
    fig, ax = plt.subplots(figsize=(8, 5))
    positions = list(range(len(seed_data)))
    bp = ax.boxplot(seed_data.values(), positions=positions, widths=0.4,
                    patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.6),
                    medianprops=dict(color='#e74c3c', linewidth=2))
    for i, (bb_label, eers_list) in enumerate(seed_data.items()):
        jitter = np.random.uniform(-0.05, 0.05, len(eers_list))
        ax.scatter([i + j for j in jitter], eers_list,
                   color='#2c3e50', zorder=5, s=50, alpha=0.8)
        mean_val = np.mean(eers_list)
        std_val = np.std(eers_list)
        ax.text(i, max(eers_list) + 0.3,
                f'{mean_val:.2f} $\\pm$ {std_val:.2f}',
                ha='center', fontsize=10, fontweight='bold')
    ax.set_xticks(positions)
    ax.set_xticklabels(seed_data.keys())
    ax.set_ylabel('ITW EER (%)')
    ax.set_title(f'Multi-Seed Training Variance (seeds: {SEEDS})')
    plt.tight_layout()
    fig_path = os.path.join(ABLATION_RESULTS_DIR, 'multi_seed_boxplot.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f'Figure saved to {fig_path}')
    plt.show()
else:
    print('No multi-seed results available.')

In [ ]:
# ============================================================================
# FIGURE 4: AUGMENTATION COMPARISON BAR CHART
# ============================================================================

aug_labels, aug_eers = [], []
for key, label, rb, mu, rir in aug_display:
    eer = all_results.get(key, {}).get('best_eer')
    if eer is not None:
        aug_labels.append(label)
        aug_eers.append(eer)

if aug_eers:
    fig, ax = plt.subplots(figsize=(8, 5))
    best_idx = np.argmin(aug_eers)
    aug_colors = ['#3498db'] * len(aug_eers)
    aug_colors[best_idx] = '#2ecc71'
    bars = ax.bar(range(len(aug_labels)), aug_eers, color=aug_colors,
                  edgecolor='white', linewidth=0.5, width=0.6)
    for bar_obj, eer_val in zip(bars, aug_eers):
        ax.text(bar_obj.get_x() + bar_obj.get_width() / 2, bar_obj.get_height() + 0.15,
                f'{eer_val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(aug_labels)))
    ax.set_xticklabels(aug_labels, rotation=15, ha='right')
    ax.set_ylabel('ITW EER (%)')
    ax.set_title('Augmentation Strategy Comparison (WavLM-Large, seed 42)')
    ax.set_ylim(0, max(aug_eers) * 1.25)
    plt.tight_layout()
    fig_path = os.path.join(ABLATION_RESULTS_DIR, 'augmentation_comparison.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f'Figure saved to {fig_path}')
    plt.show()
else:
    print('No augmentation results available.')

## 8. Summary and Next Steps

### What This Ablation Study Tells Us

1. **Per-group DA (GLADA's primary contribution)**: The `no_glada` ablation directly
   measures whether grouped-layer domain adaptation outperforms single-level global DA.
   This is the most important result for the thesis defense. MLADA (Fang et al., 2021)
   showed per-layer DA outperforms single-layer DA in vision; DAA (Ngo Trung et al., 2021)
   showed it in NLP. GLADA extends this finding to audio deepfake detection.

2. **Component contributions**: The leave-one-out ablations quantify how much each
   supporting component (VIB, speaker adversary, CKA, RawBoost, hierarchical pooling)
   contributes. Components with large $\Delta$EER are essential; those with small
   $\Delta$EER may be omitted for computational efficiency.

3. **Backbone sensitivity**: The multi-backbone sweep reveals whether GLADA's benefit
   is architecture-specific or generalizes across SSL models.

4. **Statistical reliability**: Multi-seed runs provide confidence that results are
   not due to lucky random initialization.

5. **Augmentation synergy**: Tests whether combining multiple augmentation strategies
   (RawBoost + MUSAN + RIR) improves generalization beyond any single strategy.

### Next Steps

1. **Full-epoch training**: Set `USE_FULL_EPOCHS = True` and re-run top configs with
   the full 50-epoch budget (Phase 1: 5, Phase 2: 45).
2. **Multi-dataset evaluation** (`glada_evaluation.ipynb`): Evaluate best models on
   all 8 benchmark datasets.
3. **Thesis figures** (`glada_analysis.ipynb`): Generate publication-ready LaTeX tables
   and figures from the results.

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print('=' * 60)
print('ABLATION STUDY SUMMARY')
print('=' * 60)

n_ok = sum(1 for r in all_results.values() if 'error' not in r)
n_fail = sum(1 for r in all_results.values() if 'error' in r)
print(f'  Successful runs:  {n_ok}')
print(f'  Failed runs:      {n_fail}')

best_name, best_eer_val = None, float('inf')
for name, r in all_results.items():
    if 'error' not in r and r.get('best_eer', float('inf')) < best_eer_val:
        best_eer_val = r['best_eer']
        best_name = name

if best_name:
    print(f'\n  Best config:      {best_name}')
    print(f'  Best ITW EER:     {best_eer_val:.2f}%')
    print(f'  Backbone:         {all_results[best_name].get("backbone", "N/A")}')
    print(f'  Seed:             {all_results[best_name].get("seed", "N/A")}')

full_eer_val = all_results.get('full_glada', {}).get('best_eer')
no_glada_eer = all_results.get('no_glada', {}).get('best_eer')
if full_eer_val is not None and no_glada_eer is not None:
    delta_glada = no_glada_eer - full_eer_val
    print(f'\n  KEY RESULT: Per-group DA (GLADA) vs. Global DA')
    print(f'    Full GLADA (per-group DA):   {full_eer_val:.2f}%')
    print(f'    Global DA only:              {no_glada_eer:.2f}%')
    print(f'    Delta:                       {delta_glada:+.2f}% '
          f'({"Per-group DA better" if delta_glada > 0 else "Global DA better"})')

avg_results_path = os.path.join(ABLATION_RESULTS_DIR, 'averaged_results.json')
if os.path.exists(avg_results_path):
    with open(avg_results_path, 'r') as f:
        avg_res = json.load(f)
    if avg_res:
        print(f'\n  Checkpoint Averaging Gains (top-5, Huang et al. 2025):')
        for name, ar in avg_res.items():
            single = ar.get('single_best_eer')
            averaged = ar.get('averaged_eer')
            if single is not None and averaged is not None:
                print(f'    {name}: {single:.2f}% -> {averaged:.2f}% (delta: {averaged - single:+.2f}%)')

print(f'\n  Results dir: {ABLATION_RESULTS_DIR}')
print(f'  Results JSON: {RESULTS_FILE}')

if os.path.isdir(ABLATION_RESULTS_DIR):
    fig_files = [f for f in os.listdir(ABLATION_RESULTS_DIR) if f.endswith('.png')]
    if fig_files:
        print(f'\n  Saved figures:')
        for f in sorted(fig_files):
            print(f'    {os.path.join(ABLATION_RESULTS_DIR, f)}')

print('=' * 60)
print('\nNext steps:')
print('  1. Set USE_FULL_EPOCHS = True and re-run for publication results')
print('  2. Run glada_evaluation.ipynb for multi-dataset benchmarks')
print('  3. Run glada_analysis.ipynb for thesis figures and LaTeX tables')

---

## Full Reference List

1. **Alemi et al.** "Deep Variational Information Bottleneck", ICLR 2017 -- VIB objective
2. **Clark et al.** "Electra: Pre-training Text Encoders as Discriminators", EMNLP 2020 -- LLRD
3. **El-Kheir et al.** "Layer-wise Analysis of Self-Supervised Speech Models for Deepfake Detection", NAACL 2025 -- Per-layer norm, empirical motivation for per-group DA
4. **Fang et al.** "Multi-Layer Adversarial Domain Adaptation", Neurocomputing 2021 -- MLADA (vision precedent for GLADA)
5. **Ganin et al.** "Domain-Adversarial Training of Neural Networks", JMLR 2016 -- GRL and DANN
6. **Huang et al.** "IB-CAAN", 2025 (arxiv 2509.23618) -- VIB for deepfake detection, checkpoint averaging
7. **Izmailov et al.** "Averaging Weights Leads to Wider Optima and Better Generalization", UAI 2018 -- SWA
8. **Ko et al.** "A Study on Data Augmentation of Reverberant Speech for Robust Speech Recognition", Interspeech 2017 -- RIR augmentation
9. **Kornblith et al.** "Similarity of Neural Network Representations Revisited", ICML 2019 -- CKA
10. **Liang et al.** "HierCon", WWW 2026 (arxiv 2602.01032) -- Grouped layer attention
11. **Muller et al.** "Does Audio Deepfake Detection Generalize?", Interspeech 2022 -- ITW dataset
12. **Ngo Trung et al.** "Domain Adaptation with Adapter", ACL-IJCNLP 2021 Findings -- DAA (NLP precedent for per-layer DA)
13. **Okabe et al.** "Attentive Statistics Pooling", Interspeech 2018 -- ASP
14. **Pan et al.** "Attentive Merging of Hidden Embeddings", Interspeech 2024 -- Attentive SSL layer merging
15. **Polyak & Juditsky** "Acceleration of Stochastic Approximation by Averaging", SIAM J. Control & Optimization, 1992 -- Weight averaging theory
16. **Snyder et al.** "MUSAN: A Music, Speech, and Noise Corpus", arXiv 2015 -- MUSAN corpus
17. **Tak et al.** "RawBoost", ICASSP 2022 -- Raw waveform augmentation
18. **Tran et al.** "Multi-scale Convolutional Representations", ACM MM 2025 -- CKA diversity loss
19. **Wang et al.** "ASVspoof 2019", Computer Speech & Language, 2020 -- ASVspoof 2019 LA dataset